In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lukashekiladze/walmart-competitions-utils/preprocessing.py
/kaggle/input/datasets/lukashekiladze/walmart-competitions-utils/features.py
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


In [6]:
!pip install mlflow dagshub -q

import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import dagshub
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit

In [7]:
from kaggle_secrets import UserSecretsClient

dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True)


print("MLflow tracking URI:", mlflow.get_tracking_uri())

Initialized MLflow to track repo "lshek22/walmart-recruiting-store-sales-forecasting"

Repository lshek22/walmart-recruiting-store-sales-forecasting initialized!

MLflow tracking URI: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow


In [10]:
import sys
sys.path.append("/kaggle/input/datasets/lukashekiladze/walmart-competitions-utils")

from preprocessing import load_and_clean_data
from features import generate_features

DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"

df = load_and_clean_data(DATA_DIR, include_test=True)
df = generate_features(df, lag_steps=[1, 2, 4, 52], rolling_windows=[4, 8, 12], save_plots=False)

print(df.shape)
df.head(3)

Building features
Date features done
Lag features: [1, 2, 4, 52] done
Rolling features: [4, 8, 12] done
Holiday features done
Store/Dept encoding done
Final shape: (536634, 44)
(536634, 44)


,Store,Dept,Date,Weekly_Sales,IsHoliday,IsTestSet,Type,Size,Temperature,Fuel_Price,...,holiday_weight,Holiday_Name,is_super_bowl,is_thanksgiving,is_christmas,is_labor_day,store_type_enc,store_avg_sales,dept_avg_sales,store_dept_avg_sales
0,1,1,2010-02-05,24924.50,0,0,A,151315,42.31,2.572,...,1,None,0,0,0,0,0,21710.929985,19213.485088,22513.322937
1,1,1,2010-02-12,46039.49,1,0,A,151315,38.51,2.548,...,5,Super Bowl,1,0,0,0,0,21710.929985,19213.485088,22513.322937
2,1,1,2010-02-19,41595.55,0,0,A,151315,39.93,2.514,...,1,None,0,0,0,0,0,21710.929985,19213.485088,22513.322937


In [11]:

CAT_COLS = ['Store', 'Dept', 'Type', 'store_type_enc',
            'is_super_bowl', 'is_thanksgiving', 'is_christmas', 'is_labor_day']


FEATURE_COLS = [
    'Store', 'Dept', 'store_type_enc', 'Size',
    'week_of_year', 'year', 'month', 'quarter',
    'week_sin', 'week_cos',
    'IsHoliday', 'holiday_weight',
    'is_super_bowl', 'is_thanksgiving', 'is_christmas', 'is_labor_day',
    'sales_lag_1', 'sales_lag_2', 'sales_lag_4', 'sales_lag_52',
    'rolling_mean_4', 'rolling_mean_8', 'rolling_mean_12',
    'rolling_std_4', 'rolling_std_8', 'rolling_std_12',
    'store_avg_sales', 'dept_avg_sales', 'store_dept_avg_sales',
    'Temperature', 'Fuel_Price',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'CPI', 'Unemployment',
]


train_df = df[df['IsTestSet'] == 0].dropna(subset=FEATURE_COLS + ['Weekly_Sales'])
test_df  = df[df['IsTestSet'] == 1].copy()


for col in CAT_COLS:
    train_df[col] = train_df[col].astype('category')
    test_df[col]  = test_df[col].astype('category')

X = train_df[FEATURE_COLS]
y = train_df['Weekly_Sales']
w = train_df['holiday_weight'] 

print(f"Training set: {X.shape}")
print(f"Test set:     {test_df[FEATURE_COLS].shape}")

Training set: (261083, 38)
Test set:     (115064, 38)


In [12]:
def wmae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [13]:
mlflow.set_experiment("LightGBM_Training")

with mlflow.start_run(run_name="LightGBM_Cleaning"):
    mlflow.log_params({
        "markdown_imputation": "fill_zero",
        "lag_steps":           [1, 2, 4, 52],
        "rolling_windows":     [4, 8, 12],
        "nan_strategy":        "drop_rows",
        "train_rows_after_drop": len(train_df),
    })
    mlflow.log_text(
        f"Training rows: {len(train_df)}\nFeatures: {len(FEATURE_COLS)}\n",
        "data_summary.txt"
    )
    print("cleaning run logged")

2026/07/04 09:01:32 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM_Training' does not exist. Creating a new experiment.


cleaning run logged
🏃 View run LightGBM_Cleaning at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/0/runs/5eee5c4ab5e0489eb0d76feb3af27ce0
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/0


In [14]:
train_df = train_df.sort_values('Date').reset_index(drop=True)
X = train_df[FEATURE_COLS]
y = train_df['Weekly_Sales']
w = train_df['holiday_weight']

LGB_PARAMS = {
    'objective':       'regression_l1',
    'learning_rate':   0.05,
    'num_leaves':      127,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      1,
    'verbose':          -1,
    'n_estimators':     1000,
}

tscv = TimeSeriesSplit(n_splits=3)
cv_scores = []

mlflow.set_experiment("LightGBM_Training")

with mlflow.start_run(run_name="LightGBM_CV_v1"):
    mlflow.log_params(LGB_PARAMS)
    mlflow.log_param("cv_strategy", "TimeSeriesSplit_n3")

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w_tr, w_val = w.iloc[train_idx], w.iloc[val_idx]

        model = lgb.LGBMRegressor(**LGB_PARAMS)
        model.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(100)]
        )

        preds = model.predict(X_val)
        score = wmae(y_val.values, preds, w_val.values)
        cv_scores.append(score)
        mlflow.log_metric("fold_wmae", score, step=fold)
        print(f"  Fold {fold+1} WMAE: {score:,.2f}")

    mean_wmae = np.mean(cv_scores)
    mlflow.log_metric("mean_cv_wmae", mean_wmae)
    print(f"\nMean CV WMAE: {mean_wmae:,.2f}")

[100]	valid_0's l1: 1948.53
[200]	valid_0's l1: 1870.75
  Fold 1 WMAE: 2,533.53
[100]	valid_0's l1: 1750.35
[200]	valid_0's l1: 1690.4
[300]	valid_0's l1: 1686.63
[400]	valid_0's l1: 1686.06
  Fold 2 WMAE: 1,784.57
[100]	valid_0's l1: 1324.38
[200]	valid_0's l1: 1280.31
  Fold 3 WMAE: 1,301.44

Mean CV WMAE: 1,873.18
🏃 View run LightGBM_CV_v1 at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/0/runs/5463b41a157b4dba98de276b3b39cf09
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/0


In [15]:
final_model = lgb.LGBMRegressor(**LGB_PARAMS)
final_model.fit(X, y, sample_weight=w)


pipeline = Pipeline([
    ('model', final_model)
])

with mlflow.start_run(run_name="LightGBM_Final"):
    mlflow.log_params(LGB_PARAMS)
    mlflow.log_metric("mean_cv_wmae", mean_wmae)


    mlflow.lightgbm.log_model(
        final_model,
        artifact_path="lgbm_model",
        registered_model_name="LightGBM_Walmart"
    )
    print("Model registered in MLflow Model Registry")

2026/07/04 09:04:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'LightGBM_Walmart'.
2026/07/04 09:04:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LightGBM_Walmart, version 1
Created version '1' of model 'LightGBM_Walmart'.


Model registered in MLflow Model Registry
🏃 View run LightGBM_Final at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/0/runs/6fb9f756d5814b5a9bc5125f5c1b6322
🧪 View experiment at: https://dagshub.com/lshek22/walmart-recruiting-store-sales-forecasting.mlflow/#/experiments/0


In [16]:
X_test = test_df[FEATURE_COLS]
test_preds = final_model.predict(X_test)
test_preds = np.clip(test_preds, 0, None)

submission = pd.DataFrame({
    'Id': test_df['Store'].astype(str) + '_' +
          test_df['Dept'].astype(str) + '_' +
          test_df['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': test_preds
})

submission.to_csv("submission_lgbm.csv", index=False)
print(submission.head())
print(f"Submission shape: {submission.shape}")

                 Id  Weekly_Sales
143  1_1_2012-11-02  36262.618056
144  1_1_2012-11-09   9935.708284
145  1_1_2012-11-16   9319.475935
146  1_1_2012-11-23  11614.182081
147  1_1_2012-11-30   9228.985735
Submission shape: (115064, 2)
